# Evaluación comparativa: semantic vs lexical vs hybrid

Este notebook compara **semantic search**, **lexical search** y **hybrid search** usando el mismo evidence set final. La rama híbrida fusiona exactamente las dos ramas independientes mediante Reciprocal Rank Fusion (RRF).

## Seguridad y alcance

- Este notebook es nuevo y no modifica código fuente, documentos, CSV gold ni la base de datos.
- `RUN_RETRIEVAL` empieza en `False`: no se consulta la base de datos hasta activarlo explícitamente.
- Las funciones actuales de retrieval se han inspeccionado y usan consultas `SELECT`; aun así, no se llaman funciones de ingesta ni de escritura.
- No se imprimen variables de entorno, cadenas de conexión ni secretos.
- El notebook calcula resultados en memoria. No escribe CSVs ni resultados automáticamente.

## Dos vistas del gold

1. **human_only**: solo evidencias con `review_status = human_reviewed`. Es la vista más rigurosa.
2. **expanded**: incorpora también `machine_only_unreviewed`. Aporta cobertura, pero sus resultados son análisis de sensibilidad, no gold humano.

La unidad principal es el **chunk/evidence**. Se incluyen Document y Page Hit@k como análisis secundarios.

## Protocolo de ejecución

1. Ejecuta las celdas de configuración y carga del gold.
2. Revisa que las rutas y parámetros sean correctos.
3. Cambia `RUN_RETRIEVAL` a `True` solo si autorizas las consultas de lectura y el uso del modelo de embeddings ya configurado.
4. Recupera un pool de 100 candidatos por pregunta con semantic, lexical e hybrid, usando la misma configuración léxica en la rama independiente y en la híbrida.
5. Calcula las métricas de salida a `k = 1, 3, 5, 10`.
6. Interpreta por separado las vistas `human_only` y `expanded`, tanto por pregunta como de forma agregada.
7. Ejecuta Friedman sobre una métrica primaria predefinida. Solo si es significativo, ejecuta Wilcoxon por pares con Bonferroni.

El notebook conserva las métricas por pregunta antes de agregarlas: esto permite inspección de errores y pruebas estadísticas pareadas.

In [1]:
# Importaciones. No realizan recuperación, acceso a base de datos ni escritura.
from __future__ import annotations

from dataclasses import asdict
from itertools import combinations
from pathlib import Path
import importlib.metadata
import platform
import sys

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

try:
    from scipy.stats import friedmanchisquare, wilcoxon
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print('SciPy no está disponible: las métricas funcionarán, pero no los tests estadísticos.')


In [3]:
# Configuración reproducible. Cambia valores solo antes de ejecutar retrieval.
def find_repo_root(start: Path) -> Path:
    """Busca el directorio del repositorio sin asumir desde qué carpeta se abrió Jupyter."""
    for candidate in (start, *start.parents):
        if (candidate / 'app').is_dir() and (candidate / 'evaluation_v2').is_dir():
            return candidate
    raise FileNotFoundError('No se localizó la raíz del repositorio RAGChatBot.')

ROOT = find_repo_root(Path.cwd().resolve())
FINAL_DIR = ROOT / 'evaluation_v2' / 'run_20260809_170051' / 'final_single_reviewer_v2'
QUESTIONS_PATH = ROOT / 'evaluation_v2' / 'run_20260809_170051' / 'questions.csv'
FINAL_WORKBOOK_PATH = FINAL_DIR / 'evaluation_final_single_reviewer.xlsx'

EVIDENCE_SHEET = 'Evidence Final'
CLAIMS_SHEET = 'Claims Final'
LINKS_SHEET = 'Evidence Claim Links'

# Mantener False evita consultas SELECT y llamadas al modelo de embeddings.
RUN_RETRIEVAL = True
# Los mismos cortes se aplican a las tres estrategias.
CUTOFFS = [1, 3, 5, 10]
MAX_K = max(CUTOFFS)
RETRIEVAL_DEPTH = 100  # Pool común; las métricas se calculan solo en CUTOFFS.
RRF_K = 60
TS_CONFIG = 'simple'  # Idéntica para lexical y para la rama léxica de hybrid.
TOPIC_FILTER = None  # Usa None para no aplicar un filtro que sesgue la comparación.
STRATEGIES = ['semantic', 'lexical', 'hybrid']

# Cobertura de claims: 'required_only' usa solo claims necesarios para responder completamente.
CLAIM_SCOPE = 'all_active'  # Alternativa: 'all_active' o 'required_only'

# Métrica primaria predefinida para el test estadístico.
STAT_GOLD_VIEW = 'expanded'
STAT_METRIC = 'claim_coverage_at_k'
STAT_K = 5
ALPHA = 0.05

for required_path in [QUESTIONS_PATH, FINAL_WORKBOOK_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Falta un archivo requerido: {required_path}')

print(f'Raíz: {ROOT}')
print(f'Evidence set: {FINAL_WORKBOOK_PATH.name} / {EVIDENCE_SHEET}')
print(f'RUN_RETRIEVAL = {RUN_RETRIEVAL}')


Raíz: C:\Users\mamen\Documents\Python\RAGChatBot
Evidence set: evaluation_final_single_reviewer.xlsx / Evidence Final
RUN_RETRIEVAL = True


In [4]:
# Se leen los archivos finales sin modificarlos. dtype='string' protege IDs y valores vacíos.
questions = pd.read_csv(QUESTIONS_PATH, dtype='string', keep_default_na=False)

evidence = pd.read_excel(
    FINAL_WORKBOOK_PATH,
    sheet_name=EVIDENCE_SHEET,
    dtype='string',
    keep_default_na=False,
)

claims = pd.read_excel(
    FINAL_WORKBOOK_PATH,
    sheet_name=CLAIMS_SHEET,
    dtype='string',
    keep_default_na=False,
)

links = pd.read_excel(
    FINAL_WORKBOOK_PATH,
    sheet_name=LINKS_SHEET,
    dtype='string',
    keep_default_na=False,
)

evidence['relevance_grade'] = pd.to_numeric(evidence['relevance_grade'], errors='raise')
claims['required_for_complete_answer'] = claims['required_for_complete_answer'].str.lower().eq('true')

print(f'Preguntas: {len(questions)}')
print(f'Evidencias: {len(evidence)}')
print(f'Claims activos: {len(claims)}')
display(evidence.groupby('review_status', dropna=False).size().rename('n').reset_index())


Preguntas: 31
Evidencias: 162
Claims activos: 152


,review_status,n
0,human_reviewed,40
1,machine_only_unreviewed,122


## Métricas de retrieval

Sea $r_i=1$ si el resultado en rango $i$ es una evidencia relevante y $0$ si no. En este notebook, para las métricas binarias una evidencia es relevante cuando su grado es $\geq 2$.

### Evidence Hit@k

$$Hit@k(q)=\mathbb{1}[\exists i\leq k: r_i=1]$$

Indica si la pregunta tiene al menos una evidencia útil entre los primeros $k$ resultados.

### Evidence Precision@k

$$P@k(q)=\frac{\sum_{i=1}^{k}r_i}{k}$$

Mide qué parte del contexto recuperado es útil. El denominador es siempre $k$: si se devuelven menos resultados, los rangos ausentes se tratan como no relevantes.

### Evidence Recall@k

$$Recall@k(q)=\frac{|G_q \cap R_q^{@k}|}{|G_q|}$$

$G_q$ es el conjunto de evidencias gold para la pregunta y $R_q^{@k}$ las recuperadas en top-$k$. Solo se agregan preguntas que tengan al menos una evidencia evaluable en la vista gold elegida.

### MRR

$$RR(q)=\frac{1}{\min\{i:r_i=1\}},\qquad MRR=\frac{1}{|Q|}\sum_{q\in Q}RR(q)$$

Premia encontrar la primera evidencia relevante en los primeros puestos.

### AP y MAP

$$AP@k(q)=\frac{1}{|G_q|}\sum_{i=1}^{k}P@i(q)r_i,\qquad MAP@k=\frac{1}{|Q|}\sum_{q\in Q}AP@k(q)$$

AP valora todas las evidencias relevantes y las posiciones en que aparecen; MAP es su media entre preguntas.

### Claim Coverage@k

$$ClaimCoverage@k(q)=\frac{|C_q\cap C(R_q^{@k})|}{|C_q|}$$

$C_q$ son los claims evaluables de la pregunta y $C(R_q^{@k})$ son los claims enlazados a evidencias recuperadas. Esta es la métrica más cercana a la utilidad del contexto RAG.

### Document y Page Hit@k

Usan la misma fórmula que Hit@k, pero consideran correcto respectivamente un `doc_id` gold o el par `(doc_id, page_num)` gold. Son complementarias: el análisis principal sigue siendo evidence/chunk-level.

In [5]:
# Funciones para construir las dos vistas del gold y las claves de relevancia.
def split_ids(value: str) -> set[str]:
    """Convierte listas separadas por ; o , en un conjunto de IDs sin duplicados."""
    return {part.strip() for part in str(value).replace(',', ';').split(';') if part.strip()}

def make_evidence_key(row: pd.Series) -> tuple:
    """Usa (doc_id, chunk_id); chunk_id por sí solo no tiene por qué ser globalmente único."""
    doc_id = str(row.get('doc_id', '')).strip()
    chunk_id = str(row.get('chunk_id', '')).strip()
    if chunk_id:
        return ('doc_chunk', doc_id, chunk_id)
    return ('doc_page', doc_id, str(row.get('page_num', '')).strip())

def make_gold_view(view_name: str) -> dict:
    """Prepara índices gold de evidencia, documento, página y claim para una vista."""
    if view_name == 'human_only':
        selected_evidence = evidence.loc[evidence['review_status'].eq('human_reviewed')].copy()
    elif view_name == 'expanded':
        selected_evidence = evidence.copy()
    else:
        raise ValueError("view_name debe ser 'human_only' o 'expanded'")

    selected_evidence = selected_evidence.loc[selected_evidence['relevance_grade'].ge(2)].copy()
    selected_evidence['evidence_key'] = selected_evidence.apply(make_evidence_key, axis=1)
    selected_ids = set(selected_evidence['evidence_id'])
    selected_links = links.loc[links['evidence_id'].isin(selected_ids)].copy()

    scoped_claims = claims.copy()
    if CLAIM_SCOPE == 'required_only':
        scoped_claims = scoped_claims.loc[scoped_claims['required_for_complete_answer']].copy()
    eligible_claim_ids = set(selected_links['claim_id']) & set(scoped_claims['claim_id'])
    selected_links = selected_links.loc[selected_links['claim_id'].isin(eligible_claim_ids)].copy()

    by_question = {}
    for question_id, group in selected_evidence.groupby('question_id', sort=False):
        group_links = selected_links.loc[selected_links['evidence_id'].isin(set(group['evidence_id']))]
        evidence_to_claims = group_links.groupby('evidence_id')['claim_id'].agg(lambda x: set(x)).to_dict()
        key_to_evidence_ids = group.groupby('evidence_key')['evidence_id'].agg(lambda x: set(x)).to_dict()
        evidence_to_grade = group.groupby('evidence_id')['relevance_grade'].max().to_dict()
        by_question[question_id] = {
            'gold_keys': set(group['evidence_key']),
            'gold_doc_ids': set(group['doc_id'].astype(str)),
            'gold_pages': set(zip(group['doc_id'].astype(str), group['page_num'].astype(str))),
            'key_to_evidence_ids': key_to_evidence_ids,
            'evidence_to_claims': evidence_to_claims,
            'evidence_to_grade': evidence_to_grade,
            'eligible_claim_ids': set(group_links['claim_id']),
        }
    return {'name': view_name, 'evidence': selected_evidence, 'links': selected_links, 'by_question': by_question}

gold_views = {name: make_gold_view(name) for name in ['human_only', 'expanded']}
for name, view in gold_views.items():
    evaluable_questions = sum(bool(item['gold_keys']) for item in view['by_question'].values())
    print(f'{name}: {len(view["evidence"])} evidencias, {evaluable_questions} preguntas con evidencia gold')


human_only: 40 evidencias, 18 preguntas con evidencia gold
expanded: 162 evidencias, 31 preguntas con evidencia gold


## Recuperación controlada

Las tres estrategias reciben exactamente la misma pregunta, profundidad y filtro temático. `semantic` usa solo pgvector; `lexical` usa solo PostgreSQL Full-Text Search; `hybrid` fusiona esas mismas dos ramas mediante RRF. La rama léxica independiente y la rama léxica de `hybrid` comparten `TS_CONFIG`, y la híbrida registra `RRF_K`. Los resultados se deduplican por `(doc_id, chunk_id)` antes de asignar el rango final.

La celda siguiente solo ejecuta retrieval cuando `RUN_RETRIEVAL = True`. `semantic` y `hybrid` generan el embedding de consulta; `lexical` no lo necesita. Las tres funciones realizan consultas de lectura sobre `rag_chunks`; si el modelo, PostgreSQL o la configuración FTS no están disponibles, la excepción se muestra sin ocultarla.

In [6]:
# Esta es la única celda que llama a los tres retrievers. Todas las ramas son de solo lectura.
if RUN_RETRIEVAL:
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

    # Se usa la copia revisada para comparar una ablación limpia: semantic, lexical e hybrid.
    from app.retrieval_revised_v2 import search, lexical_search, hybrid_search

    RETRIEVERS = {
        'semantic': lambda query: search(
            query, top_k=RETRIEVAL_DEPTH, topic=TOPIC_FILTER
        ),
        'lexical': lambda query: lexical_search(
            query, top_k=RETRIEVAL_DEPTH, topic=TOPIC_FILTER, ts_config=TS_CONFIG
        ),
        'hybrid': lambda query: hybrid_search(
            query, top_k=RETRIEVAL_DEPTH, topic=TOPIC_FILTER, rrf_k=RRF_K,
            candidate_k=RETRIEVAL_DEPTH, ts_config=TS_CONFIG,
        ),
    }

    retrieval_rows = []
    for question_row in questions.itertuples(index=False):
        question_id = str(question_row.question_id)
        query = str(question_row.question)
        for strategy, retrieve in RETRIEVERS.items():
            # Cada llamada devuelve datos en memoria; no se insertan ni actualizan registros.
            retrieved_chunks = retrieve(query)
            seen = set()
            final_rank = 0
            for original_rank, chunk in enumerate(retrieved_chunks, start=1):
                item = asdict(chunk)
                dedupe_key = (str(item['doc_id']), str(item['chunk_id']))
                if dedupe_key in seen:
                    continue
                seen.add(dedupe_key)
                final_rank += 1
                retrieval_rows.append({
                    'question_id': question_id,
                    'question': query,
                    'strategy': strategy,
                    'rank': final_rank,
                    'original_rank': original_rank,
                    'doc_id': item['doc_id'],
                    'chunk_id': item['chunk_id'],
                    'page_num': item['page_num'],
                    'chunk_lang': item['lang'],
                    # Los campos siguientes permiten auditar qué señal produjo cada ranking.
                    'raw_distance': item.get('distance'),
                    'raw_score': item.get('score'),
                    'semantic_rank': item.get('semantic_rank'),
                    'lexical_rank': item.get('lexical_rank'),
                    'semantic_distance': item.get('semantic_distance'),
                    'lexical_score': item.get('lexical_score'),
                    'ts_config': item.get('ts_config'),
                    'rrf_k': RRF_K if strategy == 'hybrid' else np.nan,
                    'candidate_depth': RETRIEVAL_DEPTH,
                })

    retrieval_results = pd.DataFrame(retrieval_rows)
    if retrieval_results.empty:
        raise RuntimeError('Los retrievers no devolvieron resultados; revisa el servicio y la configuración.')
    display(retrieval_results.head())
else:
    print('Retrieval no ejecutado. Cambia RUN_RETRIEVAL a True para crear retrieval_results en memoria.')


c:\Users\mamen\anaconda3\envs\RAGChatBot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,question_id,question,strategy,rank,original_rank,doc_id,chunk_id,page_num,chunk_lang,raw_distance,raw_score,semantic_rank,lexical_rank,semantic_distance,lexical_score,ts_config,rrf_k,candidate_depth
0,Q001,¿Qué es la astenia?,semantic,1,1,general_es_gepac_guia-toxicidad-quimioterapia_v1,p009_c00,9,es,0.506965,NaN,1.0,NaN,0.506965,NaN,NaN,NaN,100
1,Q001,¿Qué es la astenia?,semantic,2,2,prostata_es_gepac_guia-cancer-de-prostata_2020,p014_c00,14,es,0.527896,NaN,2.0,NaN,0.527896,NaN,NaN,NaN,100
2,Q001,¿Qué es la astenia?,semantic,3,3,general_es_gepac_guia-toxicidad-quimioterapia_v1,p004_c00,4,es,0.609280,NaN,3.0,NaN,0.609280,NaN,NaN,NaN,100
3,Q001,¿Qué es la astenia?,semantic,4,4,mama_es_hureinasofia_protocolo-cancer-mama_2021,p034_c00,34,es,0.613692,NaN,4.0,NaN,0.613692,NaN,NaN,NaN,100
4,Q001,¿Qué es la astenia?,semantic,5,5,prostata_es_gepac_guia-cancer-de-prostata_2020,p177_c00,177,es,0.626868,NaN,5.0,NaN,0.626868,NaN,NaN,NaN,100


In [7]:
retrieval_results.info()

<class 'pandas.DataFrame'>
RangeIndex: 6201 entries, 0 to 6200
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   question_id        6201 non-null   str    
 1   question           6201 non-null   str    
 2   strategy           6201 non-null   str    
 3   rank               6201 non-null   int64  
 4   original_rank      6201 non-null   int64  
 5   doc_id             6201 non-null   str    
 6   chunk_id           6201 non-null   str    
 7   page_num           6201 non-null   int64  
 8   chunk_lang         6201 non-null   str    
 9   raw_distance       3100 non-null   float64
 10  raw_score          3101 non-null   float64
 11  semantic_rank      6199 non-null   float64
 12  lexical_rank       2 non-null      float64
 13  semantic_distance  6199 non-null   float64
 14  lexical_score      2 non-null      float64
 15  ts_config          3101 non-null   str    
 16  rrf_k              3100 non-null   

In [9]:
retrieval_results_path = FINAL_DIR / 'retrieval_results_21082026_1700.csv'
retrieval_results.to_csv(retrieval_results_path, index=False)
print(f'Resultados de retrieval guardados en: {retrieval_results_path}')

Resultados de retrieval guardados en: C:\Users\mamen\Documents\Python\RAGChatBot\evaluation_v2\run_20260809_170051\final_single_reviewer_v2\retrieval_results_21082026_1700.csv


In [10]:
# Utilidades de matching: identifican chunks por (doc_id, chunk_id) y usan doc_id+página como fallback.
def result_evidence_key(row: pd.Series) -> tuple:
    doc_id = str(row.get('doc_id', '')).strip()
    chunk_id = str(row.get('chunk_id', '')).strip()
    if chunk_id:
        return ('doc_chunk', doc_id, chunk_id)
    return ('doc_page', doc_id, str(row.get('page_num', '')).strip())

def matched_evidence_ids(result_row: pd.Series, gold_for_question: dict) -> set[str]:
    """Devuelve todas las evidencias gold equivalentes a un resultado recuperado."""
    result_key = result_evidence_key(result_row)
    return set(gold_for_question['key_to_evidence_ids'].get(result_key, set()))

def evaluate_one_query(ranking: pd.DataFrame, gold_for_question: dict, cutoff: int) -> dict:
    """Calcula todas las métricas para una pregunta, estrategia y cutoff."""
    top = ranking.sort_values('rank').head(cutoff).copy()
    gold_keys = gold_for_question['gold_keys']
    gold_docs = gold_for_question['gold_doc_ids']
    gold_pages = gold_for_question['gold_pages']
    eligible_claims = gold_for_question['eligible_claim_ids']

    retrieved_gold_keys = set()
    covered_claims = set()
    binary_relevance = []

    for _, row in top.iterrows():
        evidence_ids = matched_evidence_ids(row, gold_for_question)
        is_relevant = bool(evidence_ids)
        binary_relevance.append(int(is_relevant))
        if is_relevant:
            retrieved_gold_keys.add(result_evidence_key(row))
            for evidence_id in evidence_ids:
                covered_claims.update(gold_for_question['evidence_to_claims'].get(evidence_id, set()))

    # Las posiciones no devueltas se añaden como no relevantes para mantener P@k con denominador fijo.
    binary_relevance += [0] * max(0, cutoff - len(binary_relevance))
    rel = np.asarray(binary_relevance, dtype=int)
    ranks = np.arange(1, cutoff + 1)
    precision_prefix = np.cumsum(rel) / ranks
    first_relevant = np.flatnonzero(rel)

    num_gold = len(gold_keys)
    num_claims = len(eligible_claims)
    return {
        'evaluable_evidence': num_gold > 0,
        'num_gold_evidence': num_gold,
        'num_gold_claims': num_claims,
        'evidence_hit_at_k': float(rel.any()),
        'evidence_precision_at_k': float(rel.sum() / cutoff),
        'evidence_recall_at_k': float(len(retrieved_gold_keys) / num_gold) if num_gold else np.nan,
        'mrr_at_k': float(1 / (first_relevant[0] + 1)) if len(first_relevant) else 0.0,
        'ap_at_k': float((precision_prefix * rel).sum() / num_gold) if num_gold else np.nan,
        'claim_coverage_at_k': float(len(covered_claims & eligible_claims) / num_claims) if num_claims else np.nan,
        'document_hit_at_k': float(any(str(row.doc_id) in gold_docs for row in top.itertuples(index=False))),
        'page_hit_at_k': float(any((str(row.doc_id), str(row.page_num)) in gold_pages for row in top.itertuples(index=False))),
        'retrieved_relevant_evidence': int(rel.sum()),
        'retrieved_claims': ';'.join(sorted(covered_claims & eligible_claims)),
    }


In [11]:
# Calcula métricas por pregunta y solo después agrega. Requiere ejecutar la celda de retrieval.
if 'retrieval_results' not in globals():
    raise RuntimeError('No existe retrieval_results. Activa RUN_RETRIEVAL y ejecuta la celda de recuperación primero.')

metric_rows = []
for gold_view_name, gold_view in gold_views.items():
    for question_row in questions.itertuples(index=False):
        question_id = str(question_row.question_id)
        gold_for_question = gold_view['by_question'].get(question_id)
        if gold_for_question is None:
            # No se inventan negativos cuando el gold no tiene evidencia para esa pregunta.
            continue
        question_results = retrieval_results.loc[retrieval_results['question_id'].eq(question_id)]
        # Iterar sobre STRATEGIES conserva también una estrategia que devuelva cero resultados.
        # Así no se eliminan silenciosamente consultas difíciles del promedio ni de los tests pareados.
        for strategy in STRATEGIES:
            ranking = question_results.loc[question_results['strategy'].eq(strategy)].copy()
            for cutoff in CUTOFFS:
                values = evaluate_one_query(ranking, gold_for_question, cutoff)
                metric_rows.append({
                    'gold_view': gold_view_name,
                    'question_id': question_id,
                    'strategy': strategy,
                    'k': cutoff,
                    **values,
                })

per_query_metrics = pd.DataFrame(metric_rows)
if per_query_metrics.empty:
    raise RuntimeError('No se calcularon métricas. Verifica IDs de preguntas y retrieval_results.')
display(per_query_metrics.head())


,gold_view,question_id,strategy,k,evaluable_evidence,num_gold_evidence,num_gold_claims,evidence_hit_at_k,evidence_precision_at_k,evidence_recall_at_k,mrr_at_k,ap_at_k,claim_coverage_at_k,document_hit_at_k,page_hit_at_k,retrieved_relevant_evidence,retrieved_claims
0,human_only,Q002,semantic,1,True,3,5,1.0,1.000000,0.333333,1.0,0.333333,0.4,1.0,1.0,1,Q002-C03;Q002-C05
1,human_only,Q002,semantic,3,True,3,5,1.0,0.333333,0.333333,1.0,0.333333,0.4,1.0,1.0,1,Q002-C03;Q002-C05
2,human_only,Q002,semantic,5,True,3,5,1.0,0.200000,0.333333,1.0,0.333333,0.4,1.0,1.0,1,Q002-C03;Q002-C05
3,human_only,Q002,semantic,10,True,3,5,1.0,0.100000,0.333333,1.0,0.333333,0.4,1.0,1.0,1,Q002-C03;Q002-C05
4,human_only,Q002,lexical,1,True,3,5,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0,


In [12]:
per_query_metrics_path = FINAL_DIR / 'per_query_metrics_21082026_1700.csv'
per_query_metrics.to_csv(per_query_metrics_path, index=False)
print(f'Métricas por pregunta guardadas en: {per_query_metrics_path}')

Métricas por pregunta guardadas en: C:\Users\mamen\Documents\Python\RAGChatBot\evaluation_v2\run_20260809_170051\final_single_reviewer_v2\per_query_metrics_21082026_1700.csv


In [13]:
# Agregación macro: cada pregunta evaluable pesa lo mismo. MAP es la media de AP@k.
METRIC_COLUMNS = [
    'evidence_hit_at_k', 'evidence_precision_at_k', 'evidence_recall_at_k',
    'mrr_at_k', 'ap_at_k', 'claim_coverage_at_k',
    'document_hit_at_k', 'page_hit_at_k',
]

aggregate_metrics = (
    per_query_metrics
    .groupby(['gold_view', 'strategy', 'k'], as_index=False)
    .agg(
        n_questions=('question_id', 'nunique'),
        evidence_hit_at_k=('evidence_hit_at_k', 'mean'),
        evidence_precision_at_k=('evidence_precision_at_k', 'mean'),
        evidence_recall_at_k=('evidence_recall_at_k', 'mean'),
        mrr_at_k=('mrr_at_k', 'mean'),
        map_at_k=('ap_at_k', 'mean'),
        claim_coverage_at_k=('claim_coverage_at_k', 'mean'),
        document_hit_at_k=('document_hit_at_k', 'mean'),
        page_hit_at_k=('page_hit_at_k', 'mean'),
    )
)

# Los porcentajes facilitan interpretación, sin sustituir los valores decimales de las métricas.
display(aggregate_metrics.sort_values(['gold_view', 'k', 'strategy']).style.format({
    column: '{:.3f}' for column in aggregate_metrics.columns if column not in {'gold_view', 'strategy', 'k', 'n_questions'}
}))


,gold_view,strategy,k,n_questions,evidence_hit_at_k,evidence_precision_at_k,evidence_recall_at_k,mrr_at_k,map_at_k,claim_coverage_at_k,document_hit_at_k,page_hit_at_k
0,expanded,hybrid,1,31,0.323,0.323,0.174,0.323,0.174,0.228,0.613,0.355
4,expanded,lexical,1,31,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
8,expanded,semantic,1,31,0.323,0.323,0.174,0.323,0.174,0.228,0.613,0.355
1,expanded,hybrid,3,31,0.548,0.204,0.265,0.425,0.215,0.352,0.774,0.581
5,expanded,lexical,3,31,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
9,expanded,semantic,3,31,0.548,0.204,0.265,0.430,0.220,0.352,0.774,0.581
2,expanded,hybrid,5,31,0.548,0.135,0.278,0.425,0.221,0.355,0.839,0.581
6,expanded,lexical,5,31,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
10,expanded,semantic,5,31,0.548,0.135,0.278,0.430,0.227,0.355,0.839,0.581
3,expanded,hybrid,10,31,0.677,0.094,0.377,0.442,0.239,0.488,0.903,0.710


In [14]:
aggregate_metrics_path = FINAL_DIR / 'aggregate_metrics_21082026_1700.csv'
aggregate_metrics.to_csv(aggregate_metrics_path, index=False)
print(f'Métricas agregadas guardadas en: {aggregate_metrics_path}')

Métricas agregadas guardadas en: C:\Users\mamen\Documents\Python\RAGChatBot\evaluation_v2\run_20260809_170051\final_single_reviewer_v2\aggregate_metrics_21082026_1700.csv


### Tabla comparativa de resultados

La tabla `aggregate_metrics` muestra, para cada vista gold y cada corte, una fila por sistema: `semantic`, `lexical` e `hybrid`. No se incluyen estrategias ajenas a esta comparación.

Para interpretar una ejecución concreta, compara las tres filas que tengan el mismo `gold_view` y el mismo `k`. Las conclusiones deben escribirse después de ejecutar nuevamente todo el protocolo con el Excel final actualizado.


# Es un problema de reranking?
## Resultados semánticos
Resultados expanded:

| k | Hit | Precision | Recall | MRR | MAP | Claim coverage | Doc Hit | Page Hit |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 0,323 | 0,323 | 0,174 | 0,323 | 0,174 | 0,228 | 0,613 | 0,355 |
| 5 | 0,548 | 0,135 | 0,278 | 0,430 | 0,227 | 0,355 | 0,839 | 0,581 |
| 10 | 0,677 | 0,094 | 0,377 | 0,447 | 0,244 | 0,488 | 0,903 | 0,710 |
| 20 | 0,742 | 0,055 | 0,411 | 0,451 | 0,248 | 0,509 | 0,968 | 0,774 |
| 50 | 0,806 | 0,032 | 0,521 | 0,453 | 0,254 | 0,638 | 1,000 | 0,839 |
| 100 | 0,903 | 0,022 | 0,643 | 0,454 | 0,258 | 0,743 | 1,000 | 0,935 |

Conclusiones: 
- El semántico suele localizar el documento correcto, pero no el chunk correcto.
- El principal problema no parece ser localizar el documento general, sino ordenar correctamente los fragmentos internos.
- De las 31 preguntas:
    - 10 tienen evidencia en rango 1.
    - 17 tienen evidencia en top-5.
    - 21 tienen evidencia en top-10.
    - 7 tienen la primera evidencia entre los rangos 11 y 100.
    - 3 no tienen ningún chunk gold en top-100.
- Las tres ausentes del top-100 son Q005, Q017 y Q022. Sus chunks existen en PostgreSQL, tienen contenido y embedding; por tanto, son fallos de ranking, no huecos de ingesta.

`Un reranker tiene potencial claro pero no resolvería todos los casos`

Procedimiento recomendado:
1. Recuperar 50–100 candidatos semánticos.
2. Aplicar un cross-encoder o reranker multilingüe a cada par pregunta–chunk.
3. Reordenar los candidatos.
4. Conservar los primeros 5 o 10.
5. Elegir modelo, profundidad y parámetros exclusivamente con development.
6. Ejecutar una sola evaluación final sobre test.

## Tests estadísticos pareados

Cada pregunta se evalúa con los tres sistemas; por eso las comparaciones son **pareadas**.

### Friedman

Para cada pregunta se ordenan los $m=3$ sistemas según la métrica elegida. Si $R_j$ es la suma de rangos del sistema $j$ en $n$ preguntas:

$$\chi_F^2=\frac{12}{nm(m+1)}\sum_{j=1}^{m}R_j^2-3n(m+1)$$

La hipótesis nula es que los tres sistemas tienen la misma distribución de rendimiento. Friedman no asume normalidad y es adecuado para comparaciones repetidas entre varios métodos.

### Wilcoxon signed-rank y Bonferroni

Solo si Friedman es significativo, se comparan pares. Wilcoxon ordena los valores absolutos de las diferencias por pregunta y contrasta si la diferencia típica es cero. Para tres comparaciones por pares:

$$p_{ajustado}=\min(1, 3p)$$

La corrección de Bonferroni controla el error familiar. Además de p-values, se informan diferencia media, mediana y número de preguntas donde cada estrategia gana, empata o pierde.

In [ ]:
# Friedman y Wilcoxon usan los valores por pregunta, no las medias agregadas.
if not SCIPY_AVAILABLE:
    raise ImportError('Instala/activa SciPy en el entorno antes de ejecutar los tests estadísticos.')

stat_data = per_query_metrics.loc[
    per_query_metrics['gold_view'].eq(STAT_GOLD_VIEW)
    & per_query_metrics['k'].eq(STAT_K),
    ['question_id', 'strategy', STAT_METRIC],
].dropna()

# Solo se conservan preguntas presentes en las tres columnas, condición necesaria para un test pareado.
stat_matrix = stat_data.pivot(index='question_id', columns='strategy', values=STAT_METRIC)
required_strategies = STRATEGIES
stat_matrix = stat_matrix.reindex(columns=required_strategies).dropna()

if len(stat_matrix) < 2:
    raise RuntimeError('No hay suficientes preguntas pareadas para Friedman.')

# SciPy devuelve NaN cuando todos los sistemas son idénticos en todas las preguntas.
# En ese caso la observación correcta es ausencia total de diferencias: estadístico 0, p=1.
reference = stat_matrix[required_strategies[0]].to_numpy()
all_systems_identical = all(
    np.allclose(reference, stat_matrix[name].to_numpy(), equal_nan=True)
    for name in required_strategies[1:]
)
if all_systems_identical:
    friedman_statistic, friedman_p = 0.0, 1.0
else:
    friedman_statistic, friedman_p = friedmanchisquare(
        *(stat_matrix[name] for name in required_strategies)
    )
friedman_result = pd.DataFrame([{
    'gold_view': STAT_GOLD_VIEW,
    'metric': STAT_METRIC,
    'k': STAT_K,
    'n_paired_questions': len(stat_matrix),
    'friedman_statistic': friedman_statistic,
    'p_value': friedman_p,
    'significant_at_alpha': friedman_p < ALPHA,
}])
display(friedman_result.style.format({'friedman_statistic': '{:.4f}', 'p_value': '{:.4g}'}))

if friedman_p < ALPHA:
    pair_rows = []
    pairs = list(combinations(required_strategies, 2))
    for method_a, method_b in pairs:
        values_a = stat_matrix[method_a]
        values_b = stat_matrix[method_b]
        differences = values_a - values_b

        # Wilcoxon no es definible si todos los pares son exactamente iguales.
        if np.allclose(differences.to_numpy(), 0):
            statistic, p_value = np.nan, 1.0
            test_note = 'Todas las diferencias pareadas son cero.'
        else:
            statistic, p_value = wilcoxon(values_a, values_b, alternative='two-sided', zero_method='wilcox')
            test_note = ''

        pair_rows.append({
            'method_a': method_a,
            'method_b': method_b,
            'n_paired_questions': len(differences),
            'wilcoxon_statistic': statistic,
            'p_value_raw': p_value,
            'p_value_bonferroni': min(1.0, p_value * len(pairs)),
            'mean_difference_a_minus_b': differences.mean(),
            'median_difference_a_minus_b': differences.median(),
            'a_wins': int((differences > 0).sum()),
            'ties': int((differences == 0).sum()),
            'b_wins': int((differences < 0).sum()),
            'significant_after_bonferroni': min(1.0, p_value * len(pairs)) < ALPHA,
            'note': test_note,
        })

    wilcoxon_results = pd.DataFrame(pair_rows)
    display(wilcoxon_results.style.format({
        'wilcoxon_statistic': '{:.4f}', 'p_value_raw': '{:.4g}',
        'p_value_bonferroni': '{:.4g}', 'mean_difference_a_minus_b': '{:.4f}',
        'median_difference_a_minus_b': '{:.4f}',
    }))
else:
    wilcoxon_results = pd.DataFrame()
    print('Friedman no es significativo: no se ejecutan comparaciones Wilcoxon post-hoc.')


In [ ]:
# Conserva el post-hoc en memoria. No se escribe ningún CSV.
print(f'wilcoxon_results disponible en memoria: {len(wilcoxon_results)} filas')

## Cómo interpretar los resultados

- Si **Evidence Hit@5** sube, más preguntas tienen al menos una evidencia útil en el contexto.
- Si **Precision@5** sube, hay menos ruido en el contexto.
- Si **Recall@5** o **MAP@5** suben, se recupera una mayor proporción de evidencia gold y se ordena mejor.
- Si **MRR** sube, la primera evidencia útil aparece antes.
- Si **Claim Coverage@5** sube, el contexto cubre una mayor parte de la respuesta esperada.
- Si Document Hit mejora pero Evidence Hit no, el sistema llega al documento correcto pero no al fragmento correcto.
- Si `lexical` supera a `semantic`, inspecciona si las preguntas contienen anclas exactas como nombres, códigos, cifras o términos raros.
- Si `hybrid` supera a ambas ramas, la fusión aporta información complementaria; si replica a `semantic`, revisa el solapamiento de candidatos y los rangos léxicos antes de concluir que la búsqueda léxica no ayuda.

No interpretes un p-value como tamaño de efecto. Usa conjuntamente la diferencia media por pregunta, el número de victorias/empates/derrotas y la diferencia entre `human_only` y `expanded`.

In [ ]:
# Diagnóstico por pregunta: identifica qué preguntas explican cada diferencia agregada.
def compare_strategies_per_question(gold_view: str, metric: str, k: int) -> pd.DataFrame:
    """Devuelve una matriz pregunta × estrategia para inspección y gráficos posteriores."""
    subset = per_query_metrics.loc[
        per_query_metrics['gold_view'].eq(gold_view)
        & per_query_metrics['k'].eq(k),
        ['question_id', 'strategy', metric],
    ]
    comparison = subset.pivot(index='question_id', columns='strategy', values=metric)
    # Las tres diferencias permiten saber si hybrid mejora sobre cada rama y cómo difieren las ramas base.
    comparison['hybrid_minus_semantic'] = comparison['hybrid'] - comparison['semantic']
    comparison['hybrid_minus_lexical'] = comparison['hybrid'] - comparison['lexical']
    comparison['lexical_minus_semantic'] = comparison['lexical'] - comparison['semantic']
    return comparison.sort_values(['hybrid_minus_semantic', 'hybrid_minus_lexical'])

diagnostic_table = compare_strategies_per_question(STAT_GOLD_VIEW, STAT_METRIC, STAT_K)
display(diagnostic_table)


In [ ]:
# Manifiesto no secreto para registrar el entorno de la ejecución. No escribe ningún archivo.
def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'not installed'

execution_manifest = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'pandas': package_version('pandas'),
    'numpy': package_version('numpy'),
    'scipy': package_version('scipy'),
    'run_retrieval': RUN_RETRIEVAL,
    'cutoffs': CUTOFFS,
    'retrieval_depth': RETRIEVAL_DEPTH,
    'rrf_k': RRF_K,
    'ts_config': TS_CONFIG,
    'topic_filter': TOPIC_FILTER,
    'claim_scope': CLAIM_SCOPE,
    'statistical_primary': {'gold_view': STAT_GOLD_VIEW, 'metric': STAT_METRIC, 'k': STAT_K, 'alpha': ALPHA},
    'strategies': STRATEGIES,
    'retrieval_module': 'app.retrieval_revised_v2',
    'gold_files': [str(QUESTIONS_PATH), str(FINAL_WORKBOOK_PATH)],
    'gold_sheets': [EVIDENCE_SHEET, CLAIMS_SHEET, LINKS_SHEET],
}
execution_manifest


In [ ]:
# El manifiesto permanece en memoria para respetar el alcance: solo se modifica este notebook.
display(execution_manifest)

## Referencias

- NIST TREC. *Common Evaluation Measures*: precisión, recall, MAP y métricas de ranking. https://trec.nist.gov/pubs/trec21/appendices/measures.pdf
- Järvelin, K. y Kekäläinen, J. (2002). *Cumulated Gain-Based Evaluation of IR Techniques*. https://doi.org/10.1145/582415.582418
- Smucker, M. D., Allan, J. y Carterette, B. (2007). *A Comparison of Statistical Significance Tests for Information Retrieval Evaluation*. https://maroo.cs.umass.edu/pdf/IR-591.pdf

El notebook no afirma resultados hasta que se ejecute la celda de retrieval y las celdas de métricas.